In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from ipywidgets import FloatSlider, VBox, HBox, Label, GridspecLayout, Dropdown
from IPython.display import display
import scipy.stats as st
from scipy.stats import gaussian_kde

In [2]:
df = pd.read_excel("Рубанов Михаил.xlsx", sheet_name='Sheet1')

In [3]:
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)

data = df.iloc[:, 1]
sample_data = np.random.choice(size=75000, a=data,)
print(f'\ndata has size={data.size}')


data has size=1000000


Итак, я загрузил данные, и теперь я **хочу изобразить полученные данные**, чтобы понять **какое оно имеет распределение**. Я хочу сделать это **динамически** (с возможностью выбора параметров распределения и самого распределения, поэтому, так как лагало при полных данных я буду брать случайную подвыборку от исходных данных, чтобы изобразить графики для первого задания)

In [4]:
np.random.seed(RANDOM_STATE)

# делаю графики для PDF и CDF
fig = go.FigureWidget(
    make_subplots(
        rows=1, cols=2,
        subplot_titles=['PDF', 'CDF'],
        horizontal_spacing=0.05,
    )
)

# данные для теоретических и данных из лабы
x = np.linspace(-0.05, np.max(sample_data), 1000)
sorted_data = np.sort(sample_data)
ecdf_y = (np.arange(1, len(sorted_data) + 1)) / len(sorted_data)

# настраиваю первоначальный вид
fig.add_trace(go.Scattergl(x=x, y=np.zeros_like(x), name='pdf theorethical'), row=1, col=1)
fig.add_trace(go.Histogram(x=sorted_data, histnorm='probability density', name='histogram data'), row=1, col=1)
fig.add_trace(go.Scattergl(x=x, y=np.zeros_like(x), name='cdf theorethical'), row=1, col=2)
fig.add_trace(go.Scattergl(x=sorted_data, y=ecdf_y, name='empirical cdf data'), row=1, col=2)

# делаю слайдеры для возможности изменения параметров
slider_a = FloatSlider(value=0.2,
                        min=0.001,
                        max=5.0,
                        step=0.01,
                        continuous_update=True,
                        description='a = ',)
slider_scale = FloatSlider(value=0.9,
                           min=0.001,
                           max=5.0,
                           step=0.005,
                           continuous_update=True,
                           description='scale = ',)
slider_d1 = FloatSlider(value=1.0,
                          min=0.001,
                          max=5.0,
                          step=0.05,
                          continuous_update=True,
                          description='d1 = ',)
slider_d2 = FloatSlider(value=1.0,
                          min=0.001,
                          max=5.0,
                          step=0.05,
                          continuous_update=True,
                          description='d2 = ',)
# делаю выбор распределения
dist_options = {
    'expon': st.expon,
    'gamma': st.gamma,
    'fisher': st.f,
    'weibull': st.dweibull
}

dist_dropdown = Dropdown(
    options=list(dist_options.keys()),
    value='gamma',
    description='Distribution:',
)

# функция для переключения динамического параметров и распределений
def on_slider_change(*args):
    dist_name = dist_dropdown.value
    dist = dist_options[dist_name]

    if dist_name == 'expon':
        scale = slider_scale.value
        pdf_new = dist.pdf(x, scale=scale)
        cdf_new = dist.cdf(x, scale=scale)
        slider_a.layout.visibility = 'hidden'
        slider_d1.layout.visibility = 'hidden'
        slider_d2.layout.visibility = 'hidden'

    elif dist_name == 'gamma':
        a = slider_a.value
        scale = slider_scale.value
        pdf_new = dist.pdf(x, a=a, scale=scale)
        cdf_new = dist.cdf(x, a=a, scale=scale)
        slider_a.layout.visibility = 'visible'
        slider_scale.layout.visibility = 'visible'
        slider_d1.layout.visibility = 'hidden'
        slider_d2.layout.visibility = 'hidden'

    elif dist_name == 'fisher':
        d1 = slider_d1.value
        d2 = slider_d2.value
        pdf_new = dist.pdf(x, dfn=d1, dfd=d2)
        cdf_new = dist.cdf(x, dfn=d1, dfd=d2)
        slider_a.layout.visibility = 'hidden'
        slider_scale.layout.visibility = 'hidden'
        slider_d1.layout.visibility = 'visible'
        slider_d2.layout.visibility = 'visible'
    elif dist_name == 'weibull':
        scale = slider_scale.value
        pdf_new = dist.pdf(x, c=scale)
        cdf_new = dist.cdf(x, c=scale)
        slider_a.layout.visibility = 'hidden'
        slider_scale.layout.visibility = 'visible'
        slider_d1.layout.visibility = 'hidden'
        slider_d2.layout.visibility = 'hidden'



    with fig.batch_update():
        fig.data[0].y = pdf_new
        fig.data[2].y = cdf_new
        fig.layout.annotations[0].text = f'PDF + HIST - {dist_name}'
        fig.layout.annotations[1].text = f'CDF + ECDF - {dist_name}'

# подключаю слайдеры и выпадающий список
dist_dropdown.observe(on_slider_change, names='value')
slider_a.observe(on_slider_change, names='value')
slider_scale.observe(on_slider_change, names='value')
slider_d1.observe(on_slider_change, names='value')
slider_d2.observe(on_slider_change, names='value')


fig.update_layout(hovermode='x unified',
                  margin=dict(l=10, r=10, b=20, t=50),
                  legend_orientation='h',
                  height=550)

fig.update_xaxes(matches='x', showticklabels=True, row=1, col=1)
fig.update_xaxes(matches='x', showticklabels=True, row=1, col=2)
fig.update_yaxes(matches='y', showticklabels=True, row=1, col=1)
fig.update_yaxes(matches='y', showticklabels=True, row=1, col=2)

# вывожу
display(VBox([
    HBox([dist_dropdown, slider_scale, slider_a, slider_d1, slider_d2],),
    fig
]))

on_slider_change()

fig.write_html('distribution_visualisation')

**Вывод по графикам**: как можно увидеть, у нас наши данные имеют (скорее всего) Гамма распределение с параметрами $a\approx 0.2$ и $\alpha = 0.9$